## studentAssessment(학생평가) 데이터 체크(1차 전처리)


In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

# 노트북에서 실행할 경우 프로젝트 최상위 폴더로 이동
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

STUDENT_ASSESSMENT_PATH = PROJECT_ROOT / "data" / "raw" / "studentAssessment.csv"

print("프로젝트 위치:", PROJECT_ROOT)
print("studentAssessment 존재:", STUDENT_ASSESSMENT_PATH.exists())

프로젝트 위치: C:\SKN_AI\oulad-churn-prediction
studentAssessment 존재: True


In [2]:
student_assessment = pd.read_csv(STUDENT_ASSESSMENT_PATH)

print("student_assessment 크기:", student_assessment.shape)
display(student_assessment.head())

student_assessment 크기: (173912, 5)


,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78
1,1752,28400,22,0,70
2,1752,31604,17,0,72
3,1752,32885,26,0,69
4,1752,38053,19,0,79


In [3]:
print("===== student_assessment 정보 =====")
student_assessment.info()

print("\nstudent_assessment 컬럼별 결측/공백/'?' 개수:")
for col in student_assessment.columns:
    none_count = (
        student_assessment[col].isnull()
        | (student_assessment[col].astype(str).str.strip() == "")
        | (student_assessment[col].astype(str).values == "?")
    ).sum()
    if none_count > 0:
        print(f"{col} : {none_count}")

===== student_assessment 정보 =====
<class 'pandas.DataFrame'>
RangeIndex: 173912 entries, 0 to 173911
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   id_assessment   173912 non-null  int64
 1   id_student      173912 non-null  int64
 2   date_submitted  173912 non-null  int64
 3   is_banked       173912 non-null  int64
 4   score           173912 non-null  str  
dtypes: int64(4), str(1)
memory usage: 7.0 MB

student_assessment 컬럼별 결측/공백/'?' 개수:
score : 173


`score` 결측('?') 173건은 채점이 이루어지지 않은 제출 건으로, 대체할 근거가 없어 해당 행을 제거한다.


In [4]:
# score 결측치('?') 행 제거
student_assessment_processed = student_assessment.drop(
    student_assessment[student_assessment["score"] == "?"].index
).copy()

# 제거 후 score를 숫자형으로 변환
student_assessment_processed["score"] = (
    student_assessment_processed["score"].astype(float)
)

print("제거 전 행 수:", len(student_assessment))
print("제거 후 행 수:", len(student_assessment_processed))
print("\nscore dtype:", student_assessment_processed["score"].dtype)
print("score 결측치:", student_assessment_processed["score"].isna().sum())

제거 전 행 수: 173912
제거 후 행 수: 173739

score dtype: float64
score 결측치: 0


In [5]:
STUDENT_ASSESSMENT_KEYS = ["id_assessment", "id_student"]

print("정제본 크기:", student_assessment_processed.shape)
print("\n정제본 결측치:")
print(student_assessment_processed.isna().sum())

print(
    "\n키 중복 행 수:",
    student_assessment_processed.duplicated(subset=STUDENT_ASSESSMENT_KEYS).sum()
)

print("\nscore 분포:")
print(student_assessment_processed["score"].describe())

display(student_assessment_processed.head())

정제본 크기: (173739, 5)

정제본 결측치:
id_assessment     0
id_student        0
date_submitted    0
is_banked         0
score             0
dtype: int64

키 중복 행 수: 0

score 분포:
count    173739.000000
mean         75.799573
std          18.798107
min           0.000000
25%          65.000000
50%          80.000000
75%          90.000000
max         100.000000
Name: score, dtype: float64


,id_assessment,id_student,date_submitted,is_banked,score
0,1752,11391,18,0,78.0
1,1752,28400,22,0,70.0
2,1752,31604,17,0,72.0
3,1752,32885,26,0,69.0
4,1752,38053,19,0,79.0


In [6]:
# 저장 경로
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

STUDENT_ASSESSMENT_OUTPUT_PATH = INTERIM_DIR / "student_assessment_processed.csv"

student_assessment_processed.to_csv(STUDENT_ASSESSMENT_OUTPUT_PATH, index=False)

print("저장 완료")
print(
    STUDENT_ASSESSMENT_OUTPUT_PATH.name,
    f"{STUDENT_ASSESSMENT_OUTPUT_PATH.stat().st_size / 1024**2:.2f} MB"
)

저장 완료
student_assessment_processed.csv 4.10 MB
